# Stackelberg-Duopol

We will analyze a two-firm market as the set up is given by the Stackelberg-Duopol (described in Stackelberg, H. (1934): Marktform und Gleichgewicht). We assume that both firms produce the same homogenous good. One of the firms produces first (is the Stackelberg leader). Then, the second firm produces after observing how much the leading firm produced. Since the leader firm knows that it is observed by the following firm, it incorporates this knowlegde into its decision process of how much of the good should be produced. By using subgame perfect Nash equilibrium we encounter how much of the good both firms produce in a Stackelberg-Duopol.

Later, we will extend this two firm market to an oligpol with three firms.

Imports and set magics:

In [1]:
import numpy as np
from scipy import optimize # for numerical solutions
import sympy as sm # for analytical solution
import ipywidgets as widgets # for interactive plots/buttons

# autoreload modules when code is run
%load_ext autoreload
%autoreload 2

from modelproject import stackelbergduopolClass
model = stackelbergduopolClass()

In general, we define L as the leader and F as the follower when having two firms: 
$$ L \in \{1,2\} \quad \text{and} \quad F \in \{1,2\} / \{L\}.$$ 


In the following, we assume that the first firm is the leader and the second firm is the follower. 

L chooses an amount of $x_{L}$, F chooses $x_{F}$ to produce. We assume that the two firms have different cost functions $C_{L} \neq  C_{F}$.

The Stackelberg Duopol is solved by backward induction. So, we first need to derive the best response function of the follower. 

F maximizes its protfit by: 
$$ \max_{x_{F} \geq 0} \quad P(x_{L}+ x_{F}) \cdot x_{F} - C_{F}(x_{F}) $$ 

To get the best response function of F we need to derive the profit function, $\partial/\partial x_F $, and solve it such that $x_F$ can be written as a function of $x_L$.

$$ \partial/\partial x_F = 0 \Leftrightarrow x_{F}^*  =  ... $$

L anticipates the optimal solution of F, $x_{F}^*$, and maximizes its profit by: 

$$ \max_{x_{L} \geq 0} \quad P(x_{L} +x_{F}^*) \cdot x_{L} - C_{L}(x_L)$$ 

Finally, L gets the optimal solution of $x_{L}^*$ when deriving its profit function and solving the FOC, $ \partial/\partial x_L = 0 $.

We define the inverse demand function as $\text{P}(x)$:

with $ x = x_L + x_F$:
$$

\text{P}(x) =
\begin{cases} 
 a - b\cdot(x_L + x_F), \quad \text{if} \quad (x_L + x_F) < a/b, \\
0,  \quad \text{if} \quad (x_L + x_F) \geq  a/b.
\end{cases} 
$$

We assume linear cost functions for both firms, respectively. These don't need to be exactly the same cost functions $(p_L \neq p_F)$.
$$
C_L(x_L) = p_L \cdot x_L
$$ 
$$
C_F(x_F) = p_F \cdot x_F. 
$$


## Analytical solution

There is an analytical solution for the Stackelberg Duopol. Therefore, we will first solve the model analytically by using sympy. 

We start by definining the parameters and variables in sympy: 

In [2]:
## solution using sympy 
x_L = sm.symbols("x_L")
x_F = sm.symbols("x_F")
a = sm.symbols("a")
b = sm.symbols("b")
p_L = sm.symbols("p_L")
p_F = sm.symbols("p_F")


Then we define the inverse demand function:

In [3]:
inverse_demand =  a-b*(x_L + x_F)
inverse_demand

a - b*(x_F + x_L)

Next, we define the objective function for the leading firm: 

In [4]:
objective_1 = inverse_demand * x_L - p_L*x_L
objective_1

-p_L*x_L + x_L*(a - b*(x_F + x_L))

Now, we define the objective function for the following firm:

In [5]:
objective_2 = inverse_demand * x_F - p_F*x_F
objective_2

-p_F*x_F + x_F*(a - b*(x_F + x_L))

The next step is to derive the objective function for the following firm regarding the amount $x_F$ the firm produces:

In [6]:
foc = sm.diff(objective_2, x_F)
foc

a - b*x_F - b*(x_F + x_L) - p_F

Afterwards, we solve the derivative such that $x_F$ only depends on the amount that the leader firm consumes, $x_L$:

In [7]:
sol = sm.solve(sm.Eq(foc,0), x_F)
sol ## best answer function of firm 2 

[(a - b*x_L - p_F)/(2*b)]

We substitute x_F in the objective function of the leading firm by the previous solution, $x_F^*$, such that the objective function of the leading firm only depends on $x_L$:

In [8]:
sub_objective_1= objective_1.subs(x_F, sol[0])
sub_objective_1

-p_L*x_L + x_L*(a - b*(x_L + (a - b*x_L - p_F)/(2*b)))

Finally, we can solve the FOC for the leading firm and get the optimal amount, $x_L^*$, it produces: 

In [9]:
foc_1 = sm.diff(sub_objective_1, x_L)
foc_1
sol_1 = sm.solve(sm.Eq(foc_1,0), x_L)
sol_1

[(a + p_F - 2*p_L)/(2*b)]

Now, we insert some values for the parameters a,b, $p_L$, $p_F$:
$$ a = 5 $$
$$ b = 1/4 $$ 
$$ p_L = 2 $$
$$ p_F = 1. $$

In [10]:
sol_1[0].subs(a, 5).subs(b, 1/4).subs(p_L,2).subs(p_F,1)

4.00000000000000

So the leading firm produces $x_L^*$ = 4 units of the good. 

Having $x_L^*$ = 4 we can directly find out how much the following firm will produce when it observes how much the leader produces: 

We just substitute the optimal amount of the leading firm in the best response function of the following firm: 

In [11]:
sol_2 = sol[0].subs(x_L, 4).subs(a, 5).subs(b, 1/4).subs(p_L,2).subs(p_F,1)
sol_2 

6.00000000000000

So this is the amount both firms produce when the first firm is the leading firm: 
$$ x_L^*= 4 $$
$$ x_F^*= 6 $$

## Numerical solution

We still use the same price function and cost functions. 
This is a rather brute force approach but we get the same solution as trying analytically.

We use different techniques to solve the Stacklberg Duopol numerically. 
First, use a solver from scipy. Then we use a self-defined solver (similar to the one defined in the lecture). 
At the end we try to find the root of the first derivative of the leading firm's profit function which gives the optimal solution.

Here we use scipy and a solver to find the optimal amount:

In [12]:
def constraint_x(x):
    return x  # This ensures that x is non-negative


constraints ={'type': 'ineq', 'fun': constraint_x} ## adding constraint for x


## bounds for solutions
bounds = [(0,None)] ## non negative amount x

# c. call solver, use SLSQP
initial_guess = 3
## slsqp as a method can deal with bounds und constrains
sol_case2 = optimize.minimize(
model.neg_objective_1, initial_guess, bounds = bounds ,constraints= constraints,
method='SLSQP')


In [13]:
sol_case2

 message: Optimization terminated successfully
 success: True
  status: 0
     fun: -2.0
       x: [ 4.000e+00]
     nit: 3
     jac: [ 0.000e+00]
    nfev: 6
    njev: 3

In [14]:
sol_case2.x[0]

4.0

And the solution for the second firm as the follower:

In [15]:
model.best_func2(sol_case2.x[0]) ## solution for firm 2

6.0

### Another approach to the find numerically the optimal amount:

Here, we use a self-defined solver, similar to the one in the lecture:

In [16]:
model.minimize_solver(20) ## own defined solver  

(4.0001073741824, 13, 79, 0)

It gives us the same solution as with using a solver from scipy. 

OR:

We only solve the FOC by finding the root of the first derivative of the leader's profit function.

In [17]:
## finding the root of the first derivative gives us the solution for the leading firm! 
optimize.root_scalar(model.derivative_1,x0=-5.0,method='newton')

      converged: True
           flag: converged
 function_calls: 3
     iterations: 1
           root: 4.0

The different approaches (numerically and analytically) lead to the same solution for our arbitrary parameter values. 

# Graphical analysis

We have interactive plots to visualize the difference in the optimal solution when changing the parameters of the model.

In the first plot, we only consider the optimal solutions for follower and leader.

In [18]:
# Interactive widgets
a_slider = widgets.FloatSlider(value=5, min=1, max=10, step=0.1, description='a')
b_slider = widgets.FloatSlider(value=0.25, min=0.05, max=1, step=0.05, description='b')
p1_slider = widgets.FloatSlider(value=2, min=1, max=5, step=0.1, description='p_L')
p2_slider = widgets.FloatSlider(value=1, min=0.5, max=5, step=0.1, description='p_F')

ui = widgets.VBox([p1_slider, p2_slider, a_slider, b_slider])
out = widgets.interactive_output(model.plot_stackelberg, {'p_L': p1_slider, 'p_F': p2_slider, 'a': a_slider,
                                                          'b': b_slider})

display(ui, out)


Output()

In the second plot, we also consider the profit of each firm depending on the optimal solution.

In [19]:
## plotting profit of leader and follower
widgets.interact(model.interactive_figure_profit, 
                 p1 =widgets.FloatSlider(description=r"p_L", min=0, max=3, step=0.05, value=2),
                 p2 = widgets.FloatSlider(description=r"p_F", min=0, max=3, step=0.05, value=1), 
                 a_1 =widgets.FloatSlider(description=r"a", min=2, max=10, step=0.05, value=5), 
                 b_1=widgets.FloatSlider(description=r"b", min=0, max=2, step=0.05, value=1/4));

interactive(children=(FloatSlider(value=2.0, description='p_L', max=3.0, step=0.05), FloatSlider(value=1.0, de…

# Extension


We will provide two extensions: Extension I and Extension II

In each extension we assume three firms in the market. However, in Extension I, we assume that the two following firms choose their optimal amount simultaneously. This results in a Cournot-competition for the both following firms. 

In Extension II, we allow that the following firms choose sequentially. We assume that the leader firm produces first, then the second firm and at the end the third firm. 

We will see that Extension I and Extension II lead to different solutions.

## Extension I

We will extend the market by adding another firm such that we have an oligopol. We still assume that the first firm is the leader and produces first. After that, firm 2 and firm 3 produce as followers. Now, this second stage is a Cournot competition between firm 2 and firm 3 since the following firms decide simultaneously how much to produce. Therefore, we first have to look at the FOC of the two following firms, solve the equation system such that the amount both produce can be written as function of how much the leading firm produces ($x_L$) and insert this in the profit function for the leading firm. Now the profit function of the leading firm only depends on $x_L$ and by the usual FOC we find again $x_L^*$ and afterwards $x_{F1}^*$ and $x_{F2}^*$.

F maximizes its protfit by: 
$$ \max_{x_{F} \geq 0} \quad P(x_{L}+ x_{F1} + x_{F2}) \cdot x_{F} - C_{F}(x_{F}) $$ 

To get the best response function of F1 and F2 we need to derive $\partial/\partial x_{F1}$ and  $\partial/\partial x_{F2}$ solve the two equations simultaneously such that $x_{F1}$ and $x_{F2}$ are written only as a function of $x_L$.

$$ \partial/\partial x_{F1} = 0 \Leftrightarrow x_{F1}^*  =  ... $$
$$ \partial/\partial x_{F2} = 0 \Leftrightarrow x_{F2}^*  =  ... $$

L anticipates the optimal solution of the two followers, $x_{F1}^*$ anf $x_{F2}^*$, and maximizes its profit by: 

$$ \max_{x_{L} \geq 0} \quad P(x_{L} +x_{F1}^* + x_{F2}^*) \cdot x_{L} - C_{L}(x_L)$$ 

Finally, L gets the optimal solution of $x_{L}^*$ when solving the FOC of $ \partial/\partial x_L = 0 $.

We still assume the same inverse demand function for all three firms as in the Stackelberg-Duopol.

With $$x = x_L + x_{F1} + x_{F2}$$:
$$
\text{P}(x) =
\begin{cases} 
 a - b\cdot(x_L + x_{F1} + x_{F2}), \quad \text{if} \quad (x_L + x_{F1} + x_{F2}) < a/b, \\
0,  \quad \text{if} \quad (x_L + x_{F1} + x_{F2}) \geq  a/b.
\end{cases} 
$$

### Analytical solution of the extension: 

First, we use sympy to solve the model analytically: 

In [20]:
## solution using sympy 
x_L = sm.symbols("x_L")
x_F1 = sm.symbols("x_F1")
x_F2 = sm.symbols("x_F2")
a = sm.symbols("a")
b = sm.symbols("b")
p_L = sm.symbols("p_L")
p_F1 = sm.symbols("p_F1")
p_F2 = sm.symbols("p_F2")

Since we have three firms we define inverse demand and objective functions again, including the third firm:

In [21]:
inverse_demand_extend =  a-b*(x_L + x_F1 + x_F2)
inverse_demand_extend

a - b*(x_F1 + x_F2 + x_L)

Extended objective function for the first firm (leader):

In [22]:
objective_1_extend = inverse_demand_extend * x_L - p_L*x_L
objective_1_extend

-p_L*x_L + x_L*(a - b*(x_F1 + x_F2 + x_L))

Extended objective function for the second firm (follower): 

In [23]:
objective_2_extend = inverse_demand_extend * x_F1 - p_F1*x_F1
objective_2_extend

-p_F1*x_F1 + x_F1*(a - b*(x_F1 + x_F2 + x_L))

Extended objective function for the third firm (follower):

In [24]:
objective_3_extend = inverse_demand_extend * x_F2 - p_F2*x_F2
objective_3_extend

-p_F2*x_F2 + x_F2*(a - b*(x_F1 + x_F2 + x_L))

Now we need the FOC of the second and third firm:

In [25]:
foc2_extend = sm.diff(objective_2_extend, x_F1)
foc2_extend
sol2_extend = sm.solve(sm.Eq(foc2_extend,0), x_F1)
sol2_extend

[(a - b*(x_F2 + x_L) - p_F1)/(2*b)]

In [26]:
foc3_extend = sm.diff(objective_3_extend, x_F2)
foc3_extend

a - b*x_F2 - b*(x_F1 + x_F2 + x_L) - p_F2

We substitute the solution of the second firm into the FOC of the third firm such that it only depends on x1 and x3. Afterwards, it is possible to write the amount the third firm produces only as a function of how much the leading firm produces. This can be substituted for x3 in the FOC of second firm such that it only depends on x1, too. 

In [27]:
foc3_twovariables = foc3_extend.subs(x_F1, sol2_extend[0])
foc3_twovariables

a - b*x_F2 - b*(x_F2 + x_L + (a - b*(x_F2 + x_L) - p_F1)/(2*b)) - p_F2

Here, we solve the FOC of the third firm such that the solution only depends on how much the first firm produces:

In [28]:
foc3_twovariables
solution_foc_3 = sm.solve(sm.Eq(foc3_twovariables,0), x_F2)
solution_foc_3

[(a - b*x_L + p_F1 - 2*p_F2)/(3*b)]

Now, we substitute x3 by this in the FOC of the second firm:

In [29]:
solution_foc_2 = sol2_extend[0].subs(x_F2, solution_foc_3[0])
solution_foc_2

(a - b*(x_L + (a - b*x_L + p_F1 - 2*p_F2)/(3*b)) - p_F1)/(2*b)

Finally, we can substitute x2 and x3 in the objective function of the leading firm and solve the FOC for the leader directly: 

In [30]:
objective_1_only_onevariable = objective_1_extend.subs(x_F1, solution_foc_2).subs(x_F2, solution_foc_3[0])
objective_1_only_onevariable

-p_L*x_L + x_L*(a - b*(x_L + (a - b*(x_L + (a - b*x_L + p_F1 - 2*p_F2)/(3*b)) - p_F1)/(2*b) + (a - b*x_L + p_F1 - 2*p_F2)/(3*b)))

We derive the objective function and solve the FOC of the leader:

In [31]:
foc1_extend = sm.diff(objective_1_only_onevariable, x_L)
foc1_extend

a - b*x_L/3 - b*(x_L + (a - b*(x_L + (a - b*x_L + p_F1 - 2*p_F2)/(3*b)) - p_F1)/(2*b) + (a - b*x_L + p_F1 - 2*p_F2)/(3*b)) - p_L

In [32]:
sol1_extend = sm.solve(sm.Eq(foc1_extend,0), x_L)
sol1_extend

[(a + p_F1 + p_F2 - 3*p_L)/(2*b)]

The last step is to find the amount both followers produce after having the optimal amount for the leader. We substitute x1 by the optimal x1: 

First the second firm:

In [33]:
optimal_second = solution_foc_2.subs(x_L, sol1_extend[0])

Here for the second firm:

In [34]:
optimal_third = solution_foc_3[0].subs(x_L, sol1_extend[0])

Now, we insert some, arbitrary values for the parameters a,b,c, $p_L$, $p_{F1}$, $p_{F2}$:
$$ a = 5 $$
$$ b = 1/4 $$ 
$$ p_L = 2 $$
$$ p_{F1} = 1 $$
$$ p_{F2} = 1. $$

In [35]:
sol1_extend[0].subs(a, 5).subs(b, 1/4).subs(p_L, 2).subs(p_F1,1).subs(p_F2,1)

2.00000000000000

In [36]:
optimal_second.subs(a, 5).subs(b, 1/4).subs(p_L, 2).subs(p_F1,1).subs(p_F2,1)

4.66666666666667

In [37]:
optimal_third.subs(a, 5).subs(b, 1/4).subs(p_L, 2).subs(p_F1,1).subs(p_F2,1)

4.66666666666667

### Numerical solution for Extension I:

First, we turn the objective function of the leading firm into a Python function. This allows us to use a solver to find the optimal amount for the leading firm numerically.

We have to keep in mind that solvers usually minimize. Therefore, we define the negative objective function to minimize it (same as maximizing the objective function).

In [38]:
negative_objective = -objective_1_only_onevariable

In [39]:
negative_objective_1 = negative_objective.subs(a, 5).subs(b, 1/4).subs(p_L, 2).subs(p_F1,1).subs(p_F2,1)

In [40]:
sol_func = sm.lambdify(args=(x_L),expr= negative_objective_1)
sol_func

<function _lambdifygenerated(x_L)>

In [41]:
## find the solution numerically
def constraint_x(x):
    return x  # This ensures that x is non-negative


constraints ={'type': 'ineq', 'fun': constraint_x} ## adding constraint for x


## bounds for solutions
bounds = [(0,None)] ## non negative amount x

# c. call solver, use SLSQP
initial_guess = 10
## slsqp as a method can deal with bounds und constrains
sol_case2 = optimize.minimize(
sol_func, initial_guess, bounds = bounds ,constraints= constraints,
method='SLSQP')
sol_case2

 message: Optimization terminated successfully
 success: True
  status: 0
     fun: -0.3333333333333268
       x: [ 2.000e+00]
     nit: 4
     jac: [ 0.000e+00]
    nfev: 8
    njev: 4

In [42]:
np.round(sol_case2.x,2)[0] ## same solution as before

2.0

### Visualizing Extension I

Here we visualize the model solution, again with an interactive plot.

In [43]:
## plotting profit of leader and followers
widgets.interact(model.interactive_figure_profit_extend_i, 
                 p1 =widgets.FloatSlider(description=r"p_L", min=0, max=3, step=0.05, value=2),
                 p2 = widgets.FloatSlider(description=r"p_F1", min=0, max=3, step=0.05, value=1), 
                 p3 = widgets.FloatSlider(description=r"p_F2", min=0, max=3, step=0.05, value=1.05), 
                 a_1 =widgets.FloatSlider(description=r"a", min=2, max=10, step=0.05, value=5), 
                 b_1=widgets.FloatSlider(description=r"b", min=0, max=2, step=0.05, value=1/4));

interactive(children=(FloatSlider(value=2.0, description='p_L', max=3.0, step=0.05), FloatSlider(value=1.0, de…

## Extension II

This third firm reacts to the quantities set by the first two firms:
$$
\max_{x_{F2}} \pi_{F2} = \left(P(x_L + x_{F1} + x_{F2})\right)x_{F2} - c_3 x_{F2}
$$

The first-order condition (FOC) gives:
$$
\frac{\partial \pi_{F2}}{\partial x_{F2}} = a - b x_L - b x_{F1} - 2b x_{F2} - c_3 = 0
$$

Solving for $ x_{F2}$:
$$
x_{F2} = \frac{a - b x_L - b x_{F1} - c_3}{2b}
$$

### Second Firm's Best Response (First Follower):
Knowing how $ x_{F2}$ depends on $x_L$ and $ x_{F1}$, firm 2 maximizes:
$$
\max_{x_{F1}} \pi_{F1}  \left(a - b(x_L + x_{F1} + x_{F2}(x_L, x_{F1}))\right)x_{F1} - c_2x_{F1}
$$

Plug $x_{F2}$ into the profit function, differentiate with respect to $x_{F1}$, and solve for $x_{F1}$:
$$
x_{F1} = \text{function of } x_L
$$

### First Firm's Best Response (Leader):
Finally, the leader anticipates the responses of both followers:
$$
\max_{x_L} \pi_L \left(a - b(x_L + x_2(x_L) + x_{F2}(x_L, x_{F1}(x_L)))\right)x_L - c_1x_L
$$

Differentiate and solve FOC for $x_L$:



$$
\frac{\partial \pi_L}{\partial x_L} = 0
$$

To find the optimal $x_L^*$. Then, we can plug  $x_L^*$ to the second firm's best reaction function to receive $x_{F1}^*$. Finally, we use $x_{F1}^*$ to get $x_{F2}^*$ based on the best reaction function of the last firm.

### Analytical solution of Extension II:

Again we use sympy to get the analytical solution:

In [44]:
# defining the variables and parameters
a, b, p_L, p_F1, p_F2, x_L, x_F1, x_F2 = sm.symbols('a b p_L p_F1 p_F2 x_L x_F1 x_F2')

# Third firm's best response using sympy
x_F2_expr = sm.solve(sm.diff((a - b * (x_L + x_F1 + x_F2)) * x_F2 - p_F2 * x_F2, x_F2), x_F2)[0]

# Second firm's best response using sympy
x_F1_expr = sm.solve(sm.diff((a - b * (x_L + x_F1 + x_F2_expr)) * x_F1 - p_F1 * x_F1, x_F1), x_F1)[0]

# Leader's best response using sympy
x_L_expr = sm.solve(sm.diff((a - b * (x_L + x_F1_expr + x_F2_expr.subs(x_F1, x_F1_expr))) * x_L - p_L * x_L, x_L), x_L)[0]

# inserting parameter values
param_values = {a: 5, b: 1/4, p_L: 2, p_F1: 1, p_F2: 1}

# Substituting values to find specific quantities
x_L_optimal = x_L_expr.subs(param_values)
x_F1_optimal = x_F1_expr.subs(param_values).subs(x_L, x_L_optimal)
x_F2_optimal = x_F2_expr.subs(param_values).subs({x_L: x_L_optimal, x_F1: x_F1_optimal})

print(f"Leader's optimal quantity (x_L): {x_L_optimal}")
print(f"First follower's optimal quantity (x_F1): {x_F1_optimal}")
print(f"Second follower's optimal quantity (x_F2): {x_F2_optimal}")

Leader's optimal quantity (x_L): 0
First follower's optimal quantity (x_F1): 8.00000000000000
Second follower's optimal quantity (x_F2): 4.00000000000000


### Numerical solution for Extension II:

First, we turn the objective function of the leading firm into a Python function. This allows us to use a solver to find the optimal amount for the leading firm numerically.

In [45]:
leader = ((a - b * (x_L + x_F1_expr + x_F2_expr.subs(x_F1, x_F1_expr))) * x_L - p_L * x_L)

In [46]:
leader

-p_L*x_L + x_L*(a - b*(x_L + (a - b*(x_L + (a - b*x_L - 2*p_F1 + p_F2)/(2*b)) - p_F2)/(2*b) + (a - b*x_L - 2*p_F1 + p_F2)/(2*b)))

Again for the numerical solution it is necessary to minimize the objective function:

In [47]:
min_leader = -leader

We substitute again the used values:

In [48]:
min_leader_values = min_leader.subs(a, 5).subs(b, 1/4).subs(p_L, 2).subs(p_F1,1).subs(p_F2,1)

Here, we turn the objective function of the leading firm into a Python function. This allows us to use a solver to find the optimal amount for the leading firm numerically.

In [49]:
sol_func_extend_ii = sm.lambdify(args=(x_L),expr= min_leader_values)

Now, it is possible to apply a solver: 

In [50]:
## find the solution numerically
def constraint_x(x):
    return x  # This ensures that x is non-negative


constraints ={'type': 'ineq', 'fun': constraint_x} ## adding constraint for x


## bounds for solutions
bounds = [(0,None)] ## non negative amount x

# c. call solver, use SLSQP
initial_guess = 10
## slsqp as a method can deal with bounds und constrains
sol_case3 = optimize.minimize(
sol_func_extend_ii, initial_guess, bounds = bounds ,constraints= constraints,
method='SLSQP')
sol_case3

 message: Optimization terminated successfully
 success: True
  status: 0
     fun: 0.0
       x: [ 0.000e+00]
     nit: 4
     jac: [ 9.313e-10]
    nfev: 8
    njev: 4

Our numerical solution of the second extension leads to the same result as our analytical solution. So, the leader wouldn't produce at all (in the output of sol_case3, x: [0.00]).

### Visualizing Extension II

In [51]:
## plotting profit of leader and followers
widgets.interact(model.interactive_figure_profit_extend_ii, 
                 p1 =widgets.FloatSlider(description=r"p_L", min=0, max=3, step=0.05, value=2),
                 p2 = widgets.FloatSlider(description=r"p_F1", min=0, max=3, step=0.05, value=1), 
                 p3 = widgets.FloatSlider(description=r"p_F2", min=0, max=3, step=0.05, value=1.05), 
                 a_1 =widgets.FloatSlider(description=r"a", min=2, max=10, step=0.05, value=5), 
                 b_1=widgets.FloatSlider(description=r"b", min=0, max=2, step=0.05, value=1/4));

interactive(children=(FloatSlider(value=2.0, description='p_L', max=3.0, step=0.05), FloatSlider(value=1.0, de…

# Conclusion

Our starting point was a simple Stackelberg Duopol. We solved the model analytically and numerically. Then we visualized how the optimal amount changes depending on the chosen parameters for the marginal costs and the inverse demand function.

In our setting, the follower produces more than the leader. That is due to the higher marginal costs of the leading firm. However, if both have the same cost function (as one can see in the interactive plots when changing the parameters) the leader produces more. This makes sense since we have the sequential order of production in a Stackelberg Duopol. When the leader produces first and has the same costs of production, it would produce more than the follower.

We extended the model in two different ways:

In both extensions, we extended the model by adding a third firm to the market, having an oligopol now. Solving the model gets a bit more complicated than having only a duopol. We still have one leading firm and two followers in an oligopol with three firms. 

In the first extension, we assume that the two followers will produce simultaneously.
In the second extension, we assume that the followers produce sequentially.

The solution differs in both extensions. Having a complete sequential order of production, the leader would stop producing at all with our initial (arbitrary) parameter setting. 